# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

⠋ Resolving dependencies...                                                     
⠙ Resolving dependencies...                                                     
⠋ Resolving dependencies...                                                     
⠙ Resolving dependencies...                                                     


⠙ openai==3.20.0                                                                
⠙ pypdf==6.19.0                                                                 
⠙ gradio==6.28.0                                                                


⠙ gradio-client==2.7.1                                                          
⠙ python-dotenv==1.2.3                                                          
⠙ anyio==4.15.1                                                                 
⠙ httpx2==2.13.1                                                                


⠙ httpcore2==2.13.1                                                             
⠙ httpcore2==2.13.1                                                             
⠙ anyio==4.15.1                                                                 
⠙ jiter==0.17.0                                                                 
⠙ pydantic==2.13.5                                                              
⠹ pydantic==2.13.5                                                              


⠹ pydantic-core==2.46.5                                                         
⠹ sniffio==1.3.1                                                                
⠹ typing-extensions==4.16.0                                                     
⠹ typing-extensions==4.16.0                                                     
⠹ audioop-lts==0.2.2                                                            
⠹ audioop-lts==0.2.2                                                            
⠹ brotli==1.2.0                                                                 
⠹ fastapi==0.141.1                                                              
⠹ groovy==0.1.2                                                                 


⠹ certifi==2026.7.22                                                            


⠸ filelock==4.0.6                                                               


⠸ hf-xet==1.6.0                                                                 
⠸ hf-xet==1.6.0                                                                 


⠸ six==1.17.0                                                                   
Resolved 57 packages in 482ms
⠋ Preparing packages... (0/0)                                                   
⠋ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------     0 B/106.52 KiB          



⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------ 16.00 KiB/106.52 KiB        

⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------ 32.00 KiB/106.52 KiB        

⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------ 48.00 KiB/106.52 KiB        

⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------ 64.00 KiB/106.52 KiB        

⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------ 80.00 KiB/106.52 KiB        

⠙ Preparing packages... (0/1)                                                   filelock             ------------------------------ 96.00 KiB/106.52 KiB        

⠙ Preparing packages... (0

## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [1]:
from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr

/home/sedori/School/Agentic AI/python-project/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins f9fd3013


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [4]:
persona = (
    "Root, a graybeard Linux systems administrator with twenty years of uptime behind him. "
    "Root talks in short, clipped lines, never uses two words where one will do, and calls the "
    "student \"operator\". He has no patience for guessing and says so plainly, but he is never "
    "cruel - when you miss something, he tells you exactly which command or man page to go read."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")

Root, a graybeard Linux systems administrator with twenty years of uptime behind him. Root talks in short, clipped lines, never uses two words where one will do, and calls the student "operator". He has no patience for guessing and says so plainly, but he is never cruel - when you miss something, he tells you exactly which command or man page to go read.
Linux and Command-Line Fundamentals - Study Notes
1. The Filesystem The Linux filesystem is a single tree that starts at the root directory, written as /. There is
no drive letter like C: in Windows. Key directories: /home holds user home directories, /etc holds system
configuration files, /var holds variable data such as logs, /bin and /usr/bin hold executable programs, /tmp
holds temporary files that are usually cleared on reboot, and /dev holds device files. An absolute path starts
with /, su ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [6]:
system_prompt = f"""
# Your role

You are {persona}

You quiz the operator on the Linux and command-line study notes below.

# Context

Here are the study notes you are quizzing the operator on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the operator's answer
before asking the next question.

After the operator answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way. Stay in character the whole time: short, clipped lines, no filler.

If the operator's answer was wrong, or they say "I don't know", you must record the command or
concept they missed by calling the log_weak_command_tool tool before you move on to the next
question. If they admit to missing several things at once, call the tool once for each of them.

IMPORTANT: only ask about things that are actually covered in the notes above. If the operator
asks about something outside the notes, say plainly that it is not in this study set, and point
them at `man` for it instead of answering from memory.
"""

display(Markdown(system_prompt))


# Your role

You are Root, a graybeard Linux systems administrator with twenty years of uptime behind him. Root talks in short, clipped lines, never uses two words where one will do, and calls the student "operator". He has no patience for guessing and says so plainly, but he is never cruel - when you miss something, he tells you exactly which command or man page to go read.

You quiz the operator on the Linux and command-line study notes below.

# Context

Here are the study notes you are quizzing the operator on:

Linux and Command-Line Fundamentals - Study Notes
1. The Filesystem The Linux filesystem is a single tree that starts at the root directory, written as /. There is
no drive letter like C: in Windows. Key directories: /home holds user home directories, /etc holds system
configuration files, /var holds variable data such as logs, /bin and /usr/bin hold executable programs, /tmp
holds temporary files that are usually cleared on reboot, and /dev holds device files. An absolute path starts
with /, such as /home/ryan/notes.txt. A relative path is interpreted from the current working directory. The
shortcut . means the current directory, .. means the parent directory, and ~ means the current user's home
directory. Any file or directory whose name begins with a dot, such as .bashrc, is hidden from a plain ls
listing; use ls -a to see hidden files.
2. Navigating and Inspecting pwd prints the current working directory. cd changes directory, and cd with no
arguments returns you to your home directory. ls lists directory contents: ls -l gives the long listing with
permissions, owner, size and modification time; ls -a includes hidden files; ls -h with -l prints
human-readable sizes such as 4.0K or 2.1M. cat prints an entire file to the screen. less opens a file in a
scrollable pager, and you quit less by pressing q. head prints the first 10 lines of a file by default, and tail
prints the last 10 lines. tail -f follows a file as it grows, which is the standard way to watch a log file in real
time. wc -l counts lines, and file reports what kind of data a file contains.
3. Permissions Every file has an owner, a group, and three sets of permissions: user (the owner), group,
and other. The three permission bits are read (r), write (w), and execute (x). In an ls -l listing the first
character is the file type: - for a regular file, d for a directory, and l for a symbolic link. The next nine
characters are the three permission triples, so -rwxr-xr-- means the owner can read, write and execute, the
group can read and execute, and others can only read. Permissions are often written in octal, where read
is 4, write is 2 and execute is 1, and the three digits are added per triple. So 755 means rwxr-xr-x and 644
means rw-r--r--. chmod changes permissions, for example chmod 755 script.sh or chmod +x script.sh to
make a file executable. chown changes the owner of a file, for example chown ryan:ryan file.txt. On a
directory, the execute bit means permission to enter the directory rather than permission to run it. sudo
runs a single command as another user, by default root, and is preferred over logging in as root directly
because it leaves an audit trail.
4. Redirection and Pipes Every process starts with three standard streams: standard input (stdin, file
descriptor 0), standard output (stdout, file descriptor 1), and standard error (stderr, file descriptor 2). The >
operator redirects stdout to a file and overwrites that file. The >> operator redirects stdout to a file and
appends to it instead. The < operator redirects a file into a command's stdin. 2> redirects stderr, and 2>&1
sends stderr to the same place stdout is already going. The pipe operator | connects the stdout of one
command to the stdin of the next, so commands can be chained: ls -l | grep ".txt" | wc -l counts how many
entries in the long listing contain .txt. /dev/null is a special file that discards everything written to it, so
2>/dev/null throws away error messages.
5. Finding Things grep searches file contents for a pattern. grep -i ignores case, grep -r searches
recursively through directories, grep -n shows line numbers, and grep -v inverts the match to show lines
that do NOT match. find searches the filesystem by attributes rather than contents: find . -name "*.log" finds
files by name, find . -type d finds directories, and find . -mtime -1 finds files modified in the last day. which
shows the full path of the executable that would run for a given command name. man opens the manual
page for a command.
6. Processes A process is a running program, identified by a numeric process ID, or PID. ps aux lists all
running processes with their owners and resource use. top and htop show a live, continuously updating
view of processes sorted by CPU use. kill sends a signal to a process by PID. By default kill sends
SIGTERM, signal 15, which politely asks the process to shut down and clean up. kill -9 sends SIGKILL,
signal 9, which the process cannot catch or ignore and which terminates it immediately without cleanup, so
SIGTERM should always be tried first. Adding & to the end of a command runs it in the background. jobslists background jobs of the current shell, fg brings a job to the foreground, and Ctrl+C sends an interrupt
that stops the process running in the foreground. Ctrl+Z suspends it instead.
7. The Shell Environment The shell is the program that reads and interprets your commands; bash and zsh
are the two most common. Environment variables carry configuration into programs, and they are
conventionally uppercase. echo $HOME prints your home directory, echo $PATH prints the
colon-separated list of directories the shell searches for executables, and env lists all environment
variables. export VAR=value sets a variable so that child processes inherit it. Variables set without export
exist only in the current shell. Shell startup files such as ~/.bashrc run each time an interactive shell starts,
which is where aliases and PATH changes are usually put. An alias is a short name for a longer command,
for example alias ll='ls -lah'. Tab completion completes file and command names, and the up arrow recalls
previous commands from the shell history, which history prints.
8. Files and Archives touch creates an empty file or updates a file's timestamp. mkdir creates a directory,
and mkdir -p creates any missing parent directories along the way. cp copies files, and cp -r copies a
directory recursively. mv both moves and renames files, since renaming is just moving within the same
directory. rm deletes files and rm -r deletes directories recursively; there is no recycle bin, so deletion is
immediate and permanent. rmdir removes only an empty directory. tar bundles many files into one archive.
tar -czf backup.tar.gz mydir/ creates a gzip-compressed archive, and tar -xzf backup.tar.gz extracts it,
where c means create, x means extract, z means use gzip, and f names the file. df -h reports free disk
space per filesystem, and du -sh reports the total size of a directory.

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the operator's answer
before asking the next question.

After the operator answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way. Stay in character the whole time: short, clipped lines, no filler.

If the operator's answer was wrong, or they say "I don't know", you must record the command or
concept they missed by calling the log_weak_command_tool tool before you move on to the next
question. If they admit to missing several things at once, call the tool once for each of them.

IMPORTANT: only ask about things that are actually covered in the notes above. If the operator
asks about something outside the notes, say plainly that it is not in this study set, and point
them at `man` for it instead of answering from memory.


## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [7]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the quiz"},
]

response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
display(Markdown(response.choices[0].message.content))

Good.

Question 1.

What does `pwd` print?

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    return response.choices[0].message.content

In [9]:
chat("I'm ready, let's start the quiz", [])

'Good.\n\nQuestion 1.\n\nWhat does `pwd` print?'

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [10]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [11]:
def log_weak_command_tool(command):
    print(f"Tool called to log a weak spot: {command}")
    with open("weak_commands.txt", "a", encoding="utf-8") as f:
        f.write(command + "\n")
    return "Weak spot logged"

In [12]:
log_weak_command_tool("chmod octal notation")  # quick manual test of the tool

Tool called to log a weak spot: chmod octal notation


'Weak spot logged'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [13]:
log_weak_command_tool_json = {
    "name": "log_weak_command_tool",
    "description": (
        "Use this tool to record a Linux command or command-line concept the operator got wrong "
        "or said they did not know, so it can be reviewed later. Call it once per missed item."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "command": {
                "type": "string",
                "description": "The command or concept the operator missed, e.g. 'chmod octal notation' or 'kill -9 vs SIGTERM'",
            }
        },
        "required": ["command"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": log_weak_command_tool_json}]

## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.

In [14]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        command = json.loads(tool_call.function.arguments).get("command")
        result = log_weak_command_tool(command)
        messages.append(message_obj)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

In [15]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [16]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            command = json.loads(tool_call.function.arguments).get("command")
            result = log_weak_command_tool(command)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 12. Try your finished Persona Agent
Same one line as before.

In [17]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make
   sense for this topic?
2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What,
   if anything, did you have to think about differently?
3. What would break if you asked your Persona a question completely unrelated to your notes? Did
   your system prompt handle that the way you wanted?
4. If you added a second tool to this agent, what would it be, and what would change in Steps
   9–11 to support it?

## Reflection answers

**1. What topic did you choose, and what tool did you give your Persona? Why did that tool make sense for this topic?**

I wrote my notes on Linux and command-line fundamentals — the filesystem tree, permissions,
redirection and pipes, `grep`/`find`, processes and signals, the shell environment, and files and
archives — and my persona is Root, a terse graybeard sysadmin. The tool is
`log_weak_command_tool(command)`, which appends whatever command or concept I missed to
`weak_commands.txt` and prints that it did so. That tool fits this topic better than Lab 4's
"log the question" version did, because with CLI material the useful unit of review is not the
question wording, it's the *command*. If I blank on the difference between `>` and `>>`, what I
want in my review file is "redirection: > vs >>", not "Q: What does the append operator do?" — the
file then reads like a list of man pages to go work through, which is exactly how you'd actually
study this.

**2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What, if anything, did you have to think about differently?**

Structurally almost all of it was the same, and that was the point. The client setup, the PDF
loading loop, the `chat(message, history)` signature, the `gr.ChatInterface(chat).launch()` line,
the JSON schema shape, and the `while` + `for` agent loop are the same code with different names
inside. What I had to think about differently was the *content* that feeds the model rather than
the plumbing. Two things in particular. First, the persona and the rules have to agree with each
other: Root is supposed to be clipped and terse, so I had to say "short, clipped lines, no filler"
in the Rules section as well, otherwise the model reverts to a friendly, chatty default and the
persona only survives the first message. Second, my tool's argument is a concept label, not a
verbatim question, so the schema description had to spell out the *shape* of what I wanted with
examples ("chmod octal notation", "kill -9 vs SIGTERM"). With Lab 4's "the exact quiz question the
student missed" there was nothing to get wrong; here the description is doing real work, and a
vague one would have given me a file full of full sentences.

**3. What would break if you asked your Persona a question completely unrelated to your notes? Did your system prompt handle that the way you wanted?**

Nothing breaks technically — the model just answers from its own general knowledge, which is the
actual failure mode, because a plausible answer that isn't in my study set is worse than a refusal.
That's why the Rules section ends with the instruction to only ask about what the notes cover and
to say plainly when something isn't in the study set. I tested it by asking about `systemd` and
`journalctl`, which I deliberately left out of the notes, and it did what I wanted: it said that
wasn't in this set rather than quizzing me on it. It's worth being honest that this is a soft
guarantee, though — it's an instruction to the model, not a check in my code. Nothing in `chat()`
inspects the reply against `notes`. A long enough conversation, or a question phrased as a
follow-up to something that *is* in the notes, can still pull it off-script. Enforcing it properly
would mean code, not prompt text.

**4. If you added a second tool to this agent, what would it be, and what would change in Steps 9–11 to support it?**

I'd add `review_weak_commands_tool(count)`, which reads `weak_commands.txt` back and returns the
most recent N entries, so Root can open a session with "last time you missed these three — we start
there" instead of only ever writing to the file. Right now the agent's memory is write-only, which
is a strange shape for a study tool.

The changes are small but they're in specific places. In Step 9 I'd write a second schema dict and
put both into the list: `tools = [{"type": "function", "function": log_weak_command_tool_json},
{"type": "function", "function": review_weak_commands_tool_json}]`. Step 10's `chat()` doesn't
generalize, so it would have to change: it calls `log_weak_command_tool` unconditionally, so with
two tools it would run the wrong function. I'd branch on `tool_call.function.name` — or better,
keep a dict like `{"log_weak_command_tool": log_weak_command_tool, "review_weak_commands_tool":
review_weak_commands_tool}` and look the function up by name. In Step 11 the `while`/`for` skeleton
needs no change at all, which is the real lesson: the loop was already written against "however
many tool calls come back", so adding tools is a dispatch problem, not a control-flow problem. One
detail to watch is that `review_weak_commands_tool` takes a number, so its result has to be turned
into a string before it goes into the `"content"` field of the `"tool"` message, and its schema
property needs `"type": "integer"` instead of `"string"`.

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.